# MoRoOp Dataset Walkthrough

This notebook demonstrates the public API of `moroop-dataset-toolkit`. It loads the local MoRoOp Parquet files and recreates the four publication figures without downloading data by default.

Install the package first with `python -m pip install -e .` from the repository root.

In [1]:
from pathlib import Path

from moroop_dataset_toolkit import (
    DEFAULT_HF_REPOSITORY,
    download_from_huggingface,
    download_from_kth_repository,
    load_table,
)
from moroop_dataset_toolkit.figures import (
    battery_state,
    driving_path,
    representative_gantt,
    representative_velocity,
)

DATA_DIRECTORY = Path("dataset")
SHIFT = "2026_07_27_evening"
RELEASE_TAG = "1.0.0"
FIGURES_DIRECTORY = Path("figures")

## Download a versioned release

The following calls are intentionally commented out. Uncomment exactly one after choosing the distribution source. Hugging Face supports immutable release tags; the KTH download requires the direct ZIP URL from the published repository record.

In [2]:
destination = Path("dataset")
# download_from_huggingface(
#     destination,
#     repository=DEFAULT_HF_REPOSITORY,
#     revision=RELEASE_TAG,
# )

download_from_kth_repository(
    destination,
)

DATA_DIRECTORY = destination / "data"

## Load and inspect data

Load shift-level records and the shared kit catalogue through `load_table()`.

In [3]:
jobs = load_table(DATA_DIRECTORY, "jobs", shift=SHIFT)
operations = load_table(DATA_DIRECTORY, "operations", shift=SHIFT)
robot_states = load_table(
    DATA_DIRECTORY,
    "robot_state_cleaned",
    shift=SHIFT,
    columns=["id", "created_at", "pos_x", "pos_y", "state"],
)
kits = load_table(DATA_DIRECTORY, "kits")

print(f"Hugging Face repository: {DEFAULT_HF_REPOSITORY}")
print(f"Shift: {SHIFT}")
print(f"jobs: {len(jobs):,} rows")
print(f"operations: {len(operations):,} rows")
print(f"cleaned robot states: {len(robot_states):,} rows")
print(f"kit components: {len(kits):,} rows")

jobs.head()

Hugging Face repository: Self-Organizing-Production-Logistics/mobile_robot_operations
Shift: 2026_07_27_evening
jobs: 154 rows
operations: 535 rows
cleaned robot states: 15,038 rows
kit components: 88 rows


,id,created_at,payload,operations,state,outcome,termination_reason,release_date,start_time,due_date,finish_time
0,c0ae5dde-4e2f-444b-898c-1468ada06b09,2026-07-27 16:00:02.940000+00:00,"{""job_type"": ""kit supply"", ""kit_id"": ""K001""}","[""91b767ff-9e9f-4889-8a4d-ac37f69fb88e"",""458a6...",COMPLETED,SUCCESS,NONE,2026-07-27 16:00:02.930000+00:00,2026-07-27 16:00:11.540000+00:00,2026-07-27 16:04:40.610000+00:00,2026-07-27 16:03:31.230000+00:00
1,6b75ca5f-c397-439c-a548-96dfec5d041d,2026-07-27 16:04:20.890000+00:00,"{""job_type"": ""kit supply"", ""kit_id"": ""K003""}","[""b867a8ff-450e-4831-b9b3-5ccfee036655"",""8088c...",COMPLETED,SUCCESS,NONE,2026-07-27 16:04:20.890000+00:00,2026-07-27 16:04:21.840000+00:00,2026-07-27 16:09:01.530000+00:00,2026-07-27 16:06:43.030000+00:00
2,691fa6b9-2f58-427e-9472-189759af3d87,2026-07-27 16:08:22.360000+00:00,"{""job_type"": ""kit supply"", ""kit_id"": ""K014""}","[""4494b417-9993-48d9-b57e-2792e69517b7"",""59335...",COMPLETED,SUCCESS,NONE,2026-07-27 16:08:22.360000+00:00,2026-07-27 16:08:23.340000+00:00,2026-07-27 16:13:05.550000+00:00,2026-07-27 16:11:04.940000+00:00
3,1700dcdc-550e-407d-8cd3-467235fb4a69,2026-07-27 16:11:06.080000+00:00,"{""job_type"": ""empty_box_refill""}","[""c72ac9e0-4952-483f-9ab8-502d51476d3c"",""95b20...",COMPLETED,SUCCESS,NONE,NaT,2026-07-27 16:11:07.100000+00:00,NaT,2026-07-27 16:12:53.790000+00:00
4,833527b2-8a46-4424-81a7-7a0ba8c9bf89,2026-07-27 16:12:33.660000+00:00,"{""job_type"": ""kit supply"", ""kit_id"": ""K012""}","[""9afc70cb-eb26-4014-bf6b-bb8236928599"",""6fc71...",COMPLETED,SUCCESS,NONE,2026-07-27 16:12:33.660000+00:00,2026-07-27 16:12:54.620000+00:00,2026-07-27 16:17:21.660000+00:00,2026-07-27 16:16:07.360000+00:00


In [4]:
assert list(kits.columns) == ["kit_id", "color", "size", "quantity"]
assert jobs["created_at"].is_monotonic_increasing
assert operations["created_at"].is_monotonic_increasing

kits.groupby("kit_id")["color"].nunique().rename("distinct_colors").head()

kit_id
K001    2
K002    2
K003    2
K004    2
K005    2
Name: distinct_colors, dtype: int64

## Recreate publication figures

The package functions write four PDF figures using the representative data and parameters from the publication.

In [5]:
FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)

representative_gantt(
    DATA_DIRECTORY,
    FIGURES_DIRECTORY / "representative_operations_gantt.pdf",
)
representative_velocity(
    DATA_DIRECTORY,
    FIGURES_DIRECTORY / "representative_operations_velocity.pdf",
)
battery_state(DATA_DIRECTORY, FIGURES_DIRECTORY / "robot_battery_state.pdf")
driving_path(DATA_DIRECTORY, FIGURES_DIRECTORY / "robot_driving_path.pdf")

In [6]:
expected_figures = [
    "representative_operations_gantt.pdf",
    "representative_operations_velocity.pdf",
    "robot_battery_state.pdf",
    "robot_driving_path.pdf",
]

for filename in expected_figures:
    path = FIGURES_DIRECTORY / filename
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty output: {path}"
    print(f"{filename}: {path.stat().st_size:,} bytes")

representative_operations_gantt.pdf: 25,181 bytes
representative_operations_velocity.pdf: 17,099 bytes
robot_battery_state.pdf: 18,467 bytes
robot_driving_path.pdf: 153,803 bytes


## Validation errors

The loader rejects invalid table names and enforces that shift-level tables are requested with a shift identifier.

In [7]:
for table, shift in [("not_a_table", SHIFT), ("jobs", None), ("kits", SHIFT)]:
    try:
        load_table(DATA_DIRECTORY, table, shift=shift)
    except ValueError as error:
        print(f"{table!r}, shift={shift!r}: {error}")
    else:
        raise AssertionError("Expected load_table to reject the invalid request.")

'not_a_table', shift='2026_07_27_evening': Unknown table 'not_a_table'. Expected one of ['dispatch_events', 'jobs', 'kits', 'operations', 'robot_state_cleaned', 'robot_state_raw'].
'jobs', shift=None: A shift is required to load the 'jobs' table.
'kits', shift='2026_07_27_evening': The shared kits table does not have shift-specific files.
